# Validação Externa — 2025 (Ano Endêmico)
Avalia os modelos treinados (2020–2023) no conjunto de 2025, ano endêmico.
Comparação direta com os resultados de 2024 (ano epidêmico) para medir robustez e generalização.
**Scores brutos — sem calibração aplicada.**

**Métricas prioritárias:** Sensibilidade > AUPRC > ROC-AUC > Especificidade > F1

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, average_precision_score,
    roc_auc_score, precision_score, f1_score,
    RocCurveDisplay, PrecisionRecallDisplay,
)

BASE_DIR    = '../../data/processed'
BASE_2024   = '../../data/features/baseline'
OUTPUT_MOD  = '../../output/modelos'
OUTPUT_MET  = '../../output/metricas'
OUTPUT_PLT  = '../../output/plots'
YEAR_COL    = 'year'
os.makedirs(OUTPUT_PLT, exist_ok=True)

def prep_X(df):
    df = df.copy()
    if 'age_years' in df.columns:
        df.loc[df['age_years'] > 120, 'age_years'] = np.nan
    return df.drop(columns=[YEAR_COL, 'target'], errors='ignore')

def calcular_metricas(y_true, y_pred_proba, threshold=0.5):
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'sensibilidade':  round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0,
        'especificidade': round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0,
        'auprc':          round(average_precision_score(y_true, y_pred_proba), 4),
        'roc_auc':        round(roc_auc_score(y_true, y_pred_proba), 4),
        'f1':             round(f1_score(y_true, y_pred), 4),
        'precisao':       round(precision_score(y_true, y_pred, zero_division=0), 4),
        'threshold':      threshold,
    }

## 1. Carregamento dos dados de 2025

In [ ]:
# Dataset unificado (2020-2025); filtra o ano de 2025 para validacao externa
df_treated = pd.read_parquet(os.path.join(BASE_DIR, 'dengue_treated.parquet'))
df_2025 = df_treated[df_treated[YEAR_COL] == 2025].copy()

y_2025 = df_2025['target'].dropna()
mask   = df_2025['target'].notna().values
X_2025 = prep_X(df_2025)

print(f'2025 — Total: {len(df_2025):,} | Óbitos: {int(y_2025.sum()):,} ({y_2025.mean()*100:.2f}%)')

# Referência 2024
y_2024 = pd.read_parquet(os.path.join(BASE_2024, 'y_test.parquet')).squeeze().dropna()
print(f'2024 — Total: {len(y_2024):,} | Óbitos: {int(y_2024.sum()):,} ({y_2024.mean()*100:.2f}%)')

## 2. Carregamento dos modelos

In [ ]:
MODELOS_CFG = {
    'LR (tuned)':       'logistic_regression_baseline_tuned.joblib',
    'LightGBM (tuned)': 'lightgbm_baseline_tuned.joblib',
    'XGBoost (tuned)':  'xgboost_baseline_tuned.joblib',
    'Random Forest':    'random_forest_baseline.joblib',
    'Decision Tree':    'decision_tree_baseline.joblib',
}

modelos = {}
for nome, arquivo in MODELOS_CFG.items():
    path = os.path.join(OUTPUT_MOD, arquivo)
    if os.path.exists(path):
        modelos[nome] = joblib.load(path)
    else:
        print(f'[AVISO] Modelo não encontrado, ignorado: {arquivo}')

print('Modelos carregados:', list(modelos.keys()))

## 3. Predições e métricas — 2025

In [ ]:
resultados_2025 = {}

print(f'{'Modelo':<25} {'Sensibilidade':>14} {'AUPRC':>8} {'ROC-AUC':>9} {'Especif.':>10}')
print('-' * 70)

for nome, modelo in modelos.items():
    proba = modelo.predict_proba(X_2025)[:, 1][mask]
    m     = calcular_metricas(y_2025, proba)
    resultados_2025[nome] = {'proba': proba, 'metricas': m}
    print(f'{nome:<25} {m["sensibilidade"]:>14.4f} {m["auprc"]:>8.4f} '
          f'{m["roc_auc"]:>9.4f} {m["especificidade"]:>10.4f}')

## 4. Comparação 2024 vs 2025

In [ ]:
# Carrega métricas de 2024 do CSV já salvo
df_2024 = pd.read_csv(os.path.join(OUTPUT_MET, 'COMPARACAO_TODOS_MODELOS.csv'))
df_2024 = df_2024[df_2024['modelo'].isin(modelos.keys())].set_index('modelo')

rows_comp = []
for nome, res in resultados_2025.items():
    m25 = res['metricas']
    if nome in df_2024.index:
        m24      = df_2024.loc[nome]
        sens_24  = m24['sensibilidade']
        auprc_24 = m24['auprc']
        roc_24   = m24['roc_auc']
    else:
        sens_24 = auprc_24 = roc_24 = np.nan

    rows_comp.append({
        'modelo':       nome,
        'sens_2024':    sens_24,
        'sens_2025':    m25['sensibilidade'],
        'delta_sens':   round(m25['sensibilidade'] - sens_24, 4) if not np.isnan(sens_24) else np.nan,
        'auprc_2024':   auprc_24,
        'auprc_2025':   m25['auprc'],
        'delta_auprc':  round(m25['auprc'] - auprc_24, 4) if not np.isnan(auprc_24) else np.nan,
        'roc_auc_2024': roc_24,
        'roc_auc_2025': m25['roc_auc'],
        'delta_roc':    round(m25['roc_auc'] - roc_24, 4) if not np.isnan(roc_24) else np.nan,
    })

df_comp = pd.DataFrame(rows_comp).sort_values('sens_2025', ascending=False).reset_index(drop=True)

print(f'{"Modelo":<25} {"Sens 24":>8} {"Sens 25":>8} {"Δ Sens":>8} '
      f'{"AUPRC 24":>10} {"AUPRC 25":>10} {"Δ AUPRC":>9}')
print('-' * 85)
for _, r in df_comp.iterrows():
    s24 = f'{r["sens_2024"]:>8.4f}' if not np.isnan(r['sens_2024']) else '     N/A'
    a24 = f'{r["auprc_2024"]:>10.4f}' if not np.isnan(r['auprc_2024']) else '       N/A'
    ds  = f'{r["delta_sens"]:>+8.4f}' if not np.isnan(r['delta_sens']) else '     N/A'
    da  = f'{r["delta_auprc"]:>+9.4f}' if not np.isnan(r['delta_auprc']) else '      N/A'
    print(f'{r["modelo"]:<25} {s24} {r["sens_2025"]:>8.4f} {ds} {a24} {r["auprc_2025"]:>10.4f} {da}')

## 5. Gráfico comparativo

In [ ]:
display_names = {
    'Decision Tree':    'Árvore de Decisão',
    'Random Forest':    'Random Forest',
    'LR (tuned)':       'LR (tuned)',
    'LightGBM (tuned)': 'LightGBM (tuned)',
    'XGBoost (tuned)':  'XGBoost (tuned)',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x     = np.arange(len(df_comp))
nomes = df_comp['modelo'].tolist()
w     = 0.35

for ax, (m24_col, m25_col, titulo) in zip(axes, [
    ('sens_2024',  'sens_2025',  'Sensibilidade'),
    ('auprc_2024', 'auprc_2025', 'AUPRC'),
]):
    vals_24 = df_comp[m24_col].tolist()
    vals_25 = df_comp[m25_col].tolist()

    label_24_added = False
    for i, (v24, v25) in enumerate(zip(vals_24, vals_25)):
        if not np.isnan(v24):
            ax.bar(i - w/2, v24, w, color='#C44E52', alpha=0.8,
                   label='2024' if not label_24_added else '')
            label_24_added = True
        else:
            ax.annotate('sem\n2024', xy=(i - w/2, 0.01), ha='center',
                        fontsize=7, color='#C44E52', alpha=0.7)
        ax.bar(i + w/2, v25, w, color='#4C72B0', alpha=0.8,
               label='2025' if i == 0 else '')

    ax.set_xticks(x)
    ax.set_xticklabels(
        [display_names.get(n, n) for n in nomes],
        rotation=35, ha='right', fontsize=12,
    )
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparação 2024 vs 2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'validacao_2025_comparacao.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Curvas ROC e Precision-Recall — 2025

In [ ]:
cmap   = plt.get_cmap('tab10')
colors = {nome: cmap(i) for i, nome in enumerate(modelos.keys())}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
prev_2025 = float(y_2025.mean())
axes[1].axhline(prev_2025, color='k', linestyle='--', alpha=0.4,
                label=f'Prevalência 2025 ({prev_2025:.3f})')

for nome, res in resultados_2025.items():
    RocCurveDisplay.from_predictions(
        y_2025, res['proba'], ax=axes[0], name=nome,
        color=colors[nome], lw=1.5,
    )
    PrecisionRecallDisplay.from_predictions(
        y_2025, res['proba'], ax=axes[1], name=nome,
        color=colors[nome], lw=1.5,
    )

axes[0].set_title('Curva ROC — 2025', fontweight='bold')
axes[1].set_title('Curva Precision-Recall — 2025', fontweight='bold')
for ax in axes:
    ax.legend(fontsize=7, loc='lower right' if ax == axes[0] else 'upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PLT, 'validacao_2025_curvas.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Salvamento

In [ ]:
# Métricas 2025
rows_met = []
for nome, res in resultados_2025.items():
    rows_met.append({'modelo': nome, 'dataset': '2025_endemico', **res['metricas']})

df_met_2025 = pd.DataFrame(rows_met).sort_values('auprc', ascending=False)
met_path = os.path.join(OUTPUT_MET, 'COMPARACAO_2025.csv')
df_met_2025.to_csv(met_path, index=False)
print(f'Métricas 2025 salvas: {met_path}')

# Predições 2025 por modelo
for nome, res in resultados_2025.items():
    label = nome.lower().replace(' ', '_').replace('(', '').replace(')', '')
    df_pred = pd.DataFrame({'y_true': y_2025.values, 'y_proba': res['proba']})
    pred_path = os.path.join(OUTPUT_MET, f'{label}_2025_predicoes.parquet')
    df_pred.to_parquet(pred_path, index=False)

print(f'Predições 2025 salvas em {OUTPUT_MET}/')

# Comparação 2024 vs 2025
comp_path = os.path.join(OUTPUT_MET, 'COMPARACAO_2024_vs_2025.csv')
df_comp.to_csv(comp_path, index=False)
print(f'Comparação salva: {comp_path}')